# Supplementary figures

- 1: Per-domain significant-map extent (FDR<0.05),
  VLSM vs permutation-LNM; and VLSM cross-domain Dice heatmap.
- 2: Impairment-label co-occurrence across the six
  domains — phi (Pearson on binary labels) and Jaccard (impaired-set overlap).
- 3: Lesion coverage

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *

import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Short display labels (fixed domain order = KEYS / NAMES from config)
SHORT = ["Attn/Exec", "Proc. speed", "Language", "Verbal mem.", "VS func.", "VS mem."]
n = len(KEYS)

brain, affine, bg = load_mask()

## VLSM extent and cross-domain similarity

In [ ]:
# Significance masks (F_PERM_FDR >= THR), brain-restricted, for both families
def sig_mask(fam, key):
    return load_map(fam, key, F_PERM_FDR, brain) >= THR

vlsm_masks = {k: sig_mask(FAM_VLSM, k) for k in KEYS}
lnm_masks  = {k: sig_mask(CORE_FAMILY, k) for k in KEYS}

ext_vlsm = [int(vlsm_masks[k].sum()) for k in KEYS]
ext_lnm  = [int(lnm_masks[k].sum())  for k in KEYS]

def dice(a, b):
    a = a.astype(bool) & brain
    b = b.astype(bool) & brain
    s = a.sum() + b.sum()
    return 2.0 * (a & b).sum() / s if s else np.nan

# VLSM cross-domain Dice matrix (diagonal = 1.0)
vdice = np.eye(n)
for i, ka in enumerate(KEYS):
    for j, kb in enumerate(KEYS):
        if i != j:
            vdice[i, j] = dice(vlsm_masks[ka], vlsm_masks[kb])

iu = np.triu_indices(n, 1)

# --- underlying numbers as CSVs ---
pd.DataFrame({"domain": NAMES, "vlsm_sig_voxels": ext_vlsm, "lnm_sig_voxels": ext_lnm}
             ).to_csv(FIG_DIR / "vlsm_lnm_extent.csv", index=False)
pd.DataFrame(np.round(vdice, 4), index=NAMES, columns=NAMES
             ).to_csv(FIG_DIR / "vlsm_crossdomain_dice.csv")

In [ ]:
# ---- (VLSM-specific): map extent + cross-domain Dice ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), constrained_layout=True)
fig.patch.set_facecolor("white")
x = np.arange(n)

# (a) per-domain map extent VLSM vs permutation-LNM
axa = axes[0]; w = 0.38
axa.bar(x - w/2, ext_vlsm, w, color="#ef8a62", label="VLSM")
axa.bar(x + w/2, ext_lnm,  w, color="#2166ac", label="Permutation-LNM")
axa.set_xticks(x); axa.set_xticklabels(SHORT, rotation=45, ha="right")
axa.set_ylabel("significant voxels (FDR<0.05)")
axa.set_title("Map extent: VLSM vs permutation-LNM", fontweight="bold"); axa.legend()

# (b) VLSM cross-domain Dice
axb = axes[1]
im = axb.imshow(vdice, vmin=0, vmax=1, cmap="magma", aspect="auto")
axb.set_xticks(x); axb.set_xticklabels(SHORT, rotation=45, ha="right")
axb.set_yticks(x); axb.set_yticklabels(SHORT)
for i in range(n):
    for j in range(n):
        axb.text(j, i, f"{vdice[i,j]:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if vdice[i, j] < 0.6 else "black")
axb.set_title("VLSM cross-domain Dice", fontweight="bold")
fig.colorbar(im, ax=axb, fraction=0.046, pad=0.04, label="Dice")

for ax, lab in zip(axes, "ab"):
    ax.text(-0.12, 1.04, lab, transform=ax.transAxes, fontsize=18, fontweight="bold", va="bottom")
fig.savefig(FIG_DIR / "vlsm_lnm_supplement.png", dpi=300, bbox_inches="tight")
plt.show()

## Impairment-label co-occurrence

In [ ]:
clin = pd.read_csv(DATA_DIR / "clean" / "clinical_data.csv")

L = clin[KEYS].apply(pd.to_numeric, errors="coerce")   # some columns stored as strings

In [ ]:
# 6x6 co-occurrence: phi (Pearson on binary labels) and Jaccard on impaired-patient sets
phi = np.eye(n); jac = np.eye(n)
for i in range(n):
    for j in range(i + 1, n):
        a, b = L[KEYS[i]], L[KEYS[j]]
        m = a.notna() & b.notna()
        xv, yv = a[m].values, b[m].values
        phi[i, j] = phi[j, i] = np.corrcoef(xv, yv)[0, 1]
        inter = ((xv == 1) & (yv == 1)).sum()
        union = ((xv == 1) | (yv == 1)).sum()
        jac[i, j] = jac[j, i] = inter / union if union else np.nan

pd.DataFrame(np.round(phi, 4), index=NAMES, columns=NAMES).to_csv(FIG_DIR / "quant1_label_phi_matrix.csv")
pd.DataFrame(np.round(jac, 4), index=NAMES, columns=NAMES).to_csv(FIG_DIR / "quant1_label_jaccard_matrix.csv")

iu = np.triu_indices(n, 1)
phi_off, jac_off = phi[iu], jac[iu]

In [ ]:
# ---- impairment co-occurrence heatmaps (phi + Jaccard) ----
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, M, title, cmap, vlim in [
    (axes[0], phi, "Impairment co-occurrence: phi (Pearson)", "RdBu_r", (-1, 1)),
    (axes[1], jac, "Impairment co-occurrence: Jaccard", "viridis", (0, jac[iu].max() * 1.3)),
]:
    im = ax.imshow(M, cmap=cmap, vmin=vlim[0], vmax=vlim[1])
    ax.set_xticks(range(n)); ax.set_xticklabels(SHORT, rotation=45, ha="right")
    ax.set_yticks(range(n)); ax.set_yticklabels(SHORT)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if (cmap == "viridis" and M[i, j] < vlim[1] * 0.5) else "black")
    ax.set_title(title, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
fig.savefig(FIG_DIR / "quant1_label_cooccurrence.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved", FIG_DIR / "quant1_label_cooccurrence.png")

## Lesion coverage (effect of lesion coverage)

In [ ]:
from pathlib import Path
import numpy as np, nibabel as nib, pandas as pd, matplotlib.pyplot as plt
from nilearn import plotting

brain, affine, BG = load_mask()
N_brain = int(brain.sum())
LMDIR = DATA_DIR / "raw" / "lesion_masks" / "acuteinfarct_2mm"
FREQ_NII = FIG_DIR.parent / "lesion_frequency.nii.gz"    # cached lesion-frequency map (output/)

# analysed sample = union of lesion masks across the six VLSM group comparisons
names = set()
for f in (PALM_DIR / "vlsm_volcov").glob("cognition-*_domain_score/input_paths.txt"):
    for ln in open(f):
        ln = ln.strip()
        if ln:
            names.add(Path(ln).name)
analysed = sorted(n for n in names if (LMDIR / n).exists())

# per-voxel lesion frequency (patients lesioned per voxel); cached as a NIfTI
if FREQ_NII.exists():
    freq = nib.load(FREQ_NII).get_fdata()
    print(f"loaded cached lesion-frequency map: {FREQ_NII.name}")
else:
    freq = np.zeros(brain.shape)
    for nme in analysed:
        freq += (nib.load(LMDIR / nme).get_fdata() > 0)
    nib.save(nib.Nifti1Image(freq.astype(np.float32), affine), FREQ_NII)
    print(f"computed + cached lesion-frequency map -> {FREQ_NII.name}")
print(f"analysed sample n = {len(analysed)} | peak overlap = {int(freq.max())} patients/voxel")

cov_rows = []
for k in (1, 5, 10, 20, 50):
    nvox = int(((freq >= k) & brain).sum())
    cov_rows.append({"min_patients": k, "voxels": nvox,
                     "pct_of_brain": round(100 * nvox / N_brain, 1)})
coverage = pd.DataFrame(cov_rows)
coverage.to_csv(FIG_DIR / "lesion_coverage.csv", index=False)
print(coverage.to_string(index=False))

# --- two-panel figure: (a) coverage curve | (b) lesion-frequency heatmap on slices ---
freq_img = nib.Nifti1Image(np.where(brain, freq, 0).astype(np.float32), affine)
fig = plt.figure(figsize=(13, 4.6))
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.6], wspace=0.20)

axa = fig.add_subplot(gs[0, 0])
axa.plot(coverage["min_patients"], coverage["pct_of_brain"], "o-", color="black")
axa.set_xlabel("min. patients lesioned per voxel"); axa.set_ylabel("% of brain")
axa.set_title("Lesion coverage", fontweight="bold")
for _, r in coverage.iterrows():
    axa.annotate(f"{r.pct_of_brain:.0f}%", (r.min_patients, r.pct_of_brain),
                 textcoords="offset points", xytext=(4, 6), fontsize=9)
axa.set_ylim(0, coverage["pct_of_brain"].max() * 1.13)
axa.set_xlim(coverage["min_patients"].min() - 3, coverage["min_patients"].max() + 6)

axb = fig.add_subplot(gs[0, 1])
plotting.plot_stat_map(
    freq_img, bg_img=BG, display_mode="z", cut_coords=[-18, -2, 14, 30, 46],
    axes=axb, cmap="hot", colorbar=True, threshold=1, vmax=float(freq.max()),
    annotate=False, black_bg=False, radiological=True, cbar_tick_format="%i")

fig.canvas.draw()
fig.text(axa.get_position().x0 - 0.01, 0.95, "a", fontsize=16, fontweight="bold", va="top")
fig.text(axb.get_position().x0 - 0.01, 0.95, "b", fontsize=16, fontweight="bold", va="top")

fig.savefig(FIG_DIR / "lesion_coverage.png", dpi=300, bbox_inches="tight")
plt.show()